# CSA0812 – Assignment 4
## Task 6: Design Challenge – College Fest Registration Desk

**Submitted by:** Darshan A  
**Register Number:** 1973260011  
**Marks:** 13 | **Bloom's Level:** K5 – Evaluate

### Aim
To create a Python package that processes registration files from five departments, validates records, removes duplicates, creates a clean master list, event-wise participant lists, a summary report and an error log.

### Notebook contents
- Stage 1: Inputs, outputs, rules and testable requirements
- Stage 2: Processing flow
- Stage 3: Duplicate detection, e-mail validation and exception handling
- Stage 4: Package structure, functions and pseudocode
- Stage 5: Working implementation and automatic verification


## Stage 1 – Interpret the Problem (2 Marks)

**Inputs:** Five department text files. Each line contains five comma-separated fields: register number, name, department, e-mail and events. Event names are separated by semicolons.

**Outputs:**
1. `master_list.csv` – accepted unique registrations.
2. A separate participant text file for each event.
3. `summary_report.txt` – accepted registration count and event-wise totals.
4. `error_log.txt` – damaged lines, invalid e-mails, duplicates and missing files.

**Rules and testable requirements:**
- Process every available department file.
- Each record must contain exactly five fields, and required fields must not be empty.
- E-mail must contain exactly one `@`, a non-empty local part and domain, no spaces, and a dot in the domain.
- A register number can be accepted only once; later occurrences are logged as duplicates.
- Split selected events using semicolons.
- Invalid records must not enter the master list.
- Each accepted participant must appear in every selected event file.
- A bad line or missing file must be logged without stopping the remaining processing.


## Stage 2 – Processing Flow (2 Marks)

```text
START
  |
List department files
  |
Open each available file
  |
Read one line at a time
  |
Check fields, required values, e-mail and events
  |
Is the record valid? ---- No ---> Add reason to error log
  |
 Yes
  |
Is register number already in the set?
  | Yes ---> Log duplicate
  |
  No
  |
Add register number to set and record to master list
  |
Add participant to each selected event list
  |
Continue through remaining lines and files
  |
Write master list, event files, summary and error log
  |
 END
```

Validation occurs before acceptance. Duplicate checking occurs after basic validation. Errors are recorded so the program can continue.


## Stage 3 – Critical Design Decisions (2 Marks)

**Duplicate detection:** A set named `seen_register_numbers` stores register numbers already accepted. Set membership checks are O(1) on average, so duplicate detection is efficient.

**E-mail validation:** The program uses `strip()`, `count()`, `split()` and whitespace checking. It checks for one `@`, non-empty parts, no spaces and a dot in the domain. This is a simple classroom-level check, not a complete e-mail standard validator.

**Exception handling:**
- `FileNotFoundError` – missing department file.
- `PermissionError` – file cannot be accessed.
- `UnicodeDecodeError` – file cannot be decoded as UTF-8.
- `ValueError` – invalid record fields or e-mail.

Record-level errors are caught inside the line-processing loop. File-level errors are caught around each file, allowing other department files to be processed.


## Stage 4 – Package Design (4 Marks)

### Folder structure

```text
fest_registration/
    __init__.py
    reader.py
    validation.py
    processor.py
    reports.py

fest_inputs/
    arts.txt
    science.txt
    commerce.txt
    engineering.txt
    management.txt (intentionally missing)

fest_outputs/
    master_list.csv
    event_*.txt
    summary_report.txt
    error_log.txt
```

| Module | Functions | Responsibility |
|---|---|---|
| `reader.py` | `read_department_file()` | Reads numbered lines from a file |
| `validation.py` | `validate_email()`, `parse_record()` | Checks e-mail and record fields |
| `processor.py` | `process_registrations()` | Removes duplicates and collects valid records |
| `reports.py` | `write_reports()` | Writes all output files |
| `__init__.py` | Package imports | Exposes selected functions |

### Pseudocode (SIMATS-style)

```text
BEGIN CollegeFestRegistration
    CREATE empty set SEEN
    CREATE empty list MASTER
    CREATE empty dictionary EVENTS
    CREATE empty list ERRORS

    FOR each department file
        TRY to open the file
            FOR each line with its line number
                TRY
                    SPLIT line into five fields
                    VALIDATE required fields and e-mail
                    VALIDATE event list

                    IF register number is in SEEN
                        ADD duplicate reason to ERRORS
                    ELSE
                        ADD register number to SEEN
                        ADD record to MASTER
                        FOR each selected event
                            ADD participant to EVENTS
                        END FOR
                    END IF
                CATCH record error
                    ADD file, line number and reason to ERRORS
                END TRY
            END FOR
        CATCH file error
            ADD missing-file reason to ERRORS
        END TRY
    END FOR

    WRITE master list
    WRITE event participant files
    WRITE summary report
    WRITE error log
END CollegeFestRegistration
```


## Stage 5 – Implement and Demonstrate (3 Marks)

The next cell automatically creates the sample input files, the Python package and all output folders. Four department files are created; the fifth file is deliberately missing to test error handling.

**You do not need to create or edit any files manually.**


In [1]:
from pathlib import Path
import shutil
import sys
import importlib

BASE = Path("college_fest_project")
if BASE.exists():
    shutil.rmtree(BASE)

INPUT_FOLDER = BASE / "fest_inputs"
OUTPUT_FOLDER = BASE / "fest_outputs"
PACKAGE_FOLDER = BASE / "fest_registration"

INPUT_FOLDER.mkdir(parents=True)
OUTPUT_FOLDER.mkdir(parents=True)
PACKAGE_FOLDER.mkdir(parents=True)

# Four department files. management.txt is intentionally missing.
department_data = {
    "arts.txt": (
        "192311001,Asha,Arts,asha@example.com,Music;Dance\n"
        "192311002,Bala,Arts,bala@example.com,Drama;Music\n"
    ),
    "science.txt": (
        "192311003,Charan,Science,charan@example.com,Quiz;Coding\n"
        "192311002,Bala,Science,bala2@example.com,Quiz\n"
        "192311004,Divya,Science,divyaexample.com,Quiz;Coding\n"
    ),
    "commerce.txt": (
        "192311005,Farah,Commerce,farah@example.com,Debate;Quiz\n"
        "192311006,Gokul,Commerce,gokul@example.com,Music;Debate\n"
        "DAMAGED LINE WITHOUT REQUIRED FIELDS\n"
    ),
    "engineering.txt": (
        "192311007,Harini,Engineering,harini@example.com,Coding;Robotics\n"
        "192311008,Ishan,Engineering,ishan@example.com,Robotics;Quiz\n"
    )
}

for filename, content in department_data.items():
    (INPUT_FOLDER / filename).write_text(content, encoding="utf-8")

print("Sample department files created:")
for path in sorted(INPUT_FOLDER.glob("*.txt")):
    print("-", path.name)
print("management.txt is intentionally missing.")


# Create reader.py
reader_code = '''
from pathlib import Path

def read_department_file(file_path):
    records = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if line.strip():
                records.append((line_number, line.strip()))
    return records
'''

# Create validation.py
validation_code = '''
def validate_email(email):
    email = email.strip()

    if email.count("@") != 1 or any(ch.isspace() for ch in email):
        return False

    local_part, domain = email.split("@")

    if not local_part or not domain:
        return False

    if "." not in domain or domain.startswith(".") or domain.endswith("."):
        return False

    return True


def parse_record(line):
    parts = [part.strip() for part in line.split(",")]

    if len(parts) != 5:
        raise ValueError("Expected exactly 5 fields")

    register_number, name, department, email, event_text = parts

    if not register_number or not name or not department or not event_text:
        raise ValueError("A required field is empty")

    if not validate_email(email):
        raise ValueError(f"Invalid e-mail address: {email}")

    events = [event.strip() for event in event_text.split(";") if event.strip()]

    if not events:
        raise ValueError("No valid events were selected")

    return register_number, name, department, email, events
'''

# Create processor.py
processor_code = '''
from pathlib import Path
from .reader import read_department_file
from .validation import parse_record

def process_registrations(input_folder, department_files):
    input_folder = Path(input_folder)
    seen_register_numbers = set()
    master_records = []
    event_participants = {}
    errors = []

    for filename in department_files:
        file_path = input_folder / filename

        try:
            numbered_lines = read_department_file(file_path)

            for line_number, line in numbered_lines:
                try:
                    record = parse_record(line)
                    register_number, name, department, email, events = record

                    if register_number in seen_register_numbers:
                        errors.append(
                            f"{filename} | Line {line_number} | "
                            f"Duplicate register number: {register_number}"
                        )
                        continue

                    seen_register_numbers.add(register_number)
                    master_records.append(record)

                    for event in events:
                        event_participants.setdefault(event, []).append(
                            (register_number, name, department)
                        )

                except ValueError as error:
                    errors.append(
                        f"{filename} | Line {line_number} | {line} | {error}"
                    )

        except FileNotFoundError:
            errors.append(f"{filename} | File not found")
        except PermissionError:
            errors.append(f"{filename} | Permission denied")
        except UnicodeDecodeError:
            errors.append(f"{filename} | Could not decode file as UTF-8")

    return master_records, event_participants, errors
'''

# Create reports.py
reports_code = '''
from pathlib import Path
import csv

def write_reports(master_records, event_participants, errors, output_folder):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    master_path = output_folder / "master_list.csv"
    with open(master_path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["Register Number", "Name", "Department", "Email", "Events"])
        for register_number, name, department, email, events in master_records:
            writer.writerow([register_number, name, department, email, ";".join(events)])

    for event, participants in sorted(event_participants.items()):
        safe_name = "".join(
            ch.lower() if ch.isalnum() else "_" for ch in event
        ).strip("_")
        event_path = output_folder / f"event_{safe_name}.txt"

        with open(event_path, "w", encoding="utf-8") as file:
            file.write(f"PARTICIPANTS - {event}\\n")
            file.write("========================\\n")
            for register_number, name, department in participants:
                file.write(f"{register_number}, {name}, {department}\\n")

    summary_path = output_folder / "summary_report.txt"
    with open(summary_path, "w", encoding="utf-8") as file:
        file.write("COLLEGE FEST REGISTRATION SUMMARY\\n")
        file.write("=================================\\n")
        file.write(f"Accepted unique registrations: {len(master_records)}\\n")
        file.write(f"Events conducted: {len(event_participants)}\\n")
        file.write("\\nEvent-wise participant totals\\n")
        for event, participants in sorted(event_participants.items()):
            file.write(f"{event}: {len(participants)}\\n")
        file.write(f"Total errors recorded: {len(errors)}\\n")

    error_path = output_folder / "error_log.txt"
    with open(error_path, "w", encoding="utf-8") as file:
        file.write("COLLEGE FEST ERROR LOG\\n")
        file.write("======================\\n")
        if errors:
            for error in errors:
                file.write(error + "\\n")
        else:
            file.write("No errors found.\\n")

    return master_path, summary_path, error_path
'''

init_code = '''
from .reader import read_department_file
from .validation import validate_email, parse_record
from .processor import process_registrations
from .reports import write_reports
'''

(PACKAGE_FOLDER / "__init__.py").write_text(init_code, encoding="utf-8")
(PACKAGE_FOLDER / "reader.py").write_text(reader_code, encoding="utf-8")
(PACKAGE_FOLDER / "validation.py").write_text(validation_code, encoding="utf-8")
(PACKAGE_FOLDER / "processor.py").write_text(processor_code, encoding="utf-8")
(PACKAGE_FOLDER / "reports.py").write_text(reports_code, encoding="utf-8")

sys.path.insert(0, str(BASE.resolve()))
for module_name in list(sys.modules):
    if module_name == "fest_registration" or module_name.startswith("fest_registration."):
        del sys.modules[module_name]

from fest_registration import process_registrations, write_reports

department_files = [
    "arts.txt", "science.txt", "commerce.txt",
    "engineering.txt", "management.txt"
]

master_records, event_participants, errors = process_registrations(
    INPUT_FOLDER, department_files
)

master_path, summary_path, error_path = write_reports(
    master_records, event_participants, errors, OUTPUT_FOLDER
)

print("\nPROCESSING RESULT")
print("-----------------")
print("Accepted unique registrations:", len(master_records))
print("Events processed:", len(event_participants))
print("Errors recorded:", len(errors))
print("Master list saved to:", master_path)
print("Summary report saved to:", summary_path)
print("Error log saved to:", error_path)


Sample department files created:
- arts.txt
- commerce.txt
- engineering.txt
- science.txt
management.txt is intentionally missing.

PROCESSING RESULT
-----------------
Accepted unique registrations: 7
Events processed: 7
Errors recorded: 4
Master list saved to: college_fest_project/fest_outputs/master_list.csv
Summary report saved to: college_fest_project/fest_outputs/summary_report.txt
Error log saved to: college_fest_project/fest_outputs/error_log.txt


## Display the Generated Output Files

This cell displays the clean master list, event-wise participant files, summary report and error log.


In [2]:
print("CLEAN MASTER LIST")
print("=================")
print((OUTPUT_FOLDER / "master_list.csv").read_text(encoding="utf-8"))

print("\nEVENT-WISE PARTICIPANT FILES")
print("============================")
for event_file in sorted(OUTPUT_FOLDER.glob("event_*.txt")):
    print("\n" + event_file.name)
    print(event_file.read_text(encoding="utf-8"))

print("\nSUMMARY REPORT")
print("==============")
print((OUTPUT_FOLDER / "summary_report.txt").read_text(encoding="utf-8"))

print("\nERROR LOG")
print("=========")
print((OUTPUT_FOLDER / "error_log.txt").read_text(encoding="utf-8"))


CLEAN MASTER LIST
Register Number,Name,Department,Email,Events
192311001,Asha,Arts,asha@example.com,Music;Dance
192311002,Bala,Arts,bala@example.com,Drama;Music
192311003,Charan,Science,charan@example.com,Quiz;Coding
192311005,Farah,Commerce,farah@example.com,Debate;Quiz
192311006,Gokul,Commerce,gokul@example.com,Music;Debate
192311007,Harini,Engineering,harini@example.com,Coding;Robotics
192311008,Ishan,Engineering,ishan@example.com,Robotics;Quiz


EVENT-WISE PARTICIPANT FILES

event_coding.txt
PARTICIPANTS - Coding
192311003, Charan, Science
192311007, Harini, Engineering


event_dance.txt
PARTICIPANTS - Dance
192311001, Asha, Arts


event_debate.txt
PARTICIPANTS - Debate
192311005, Farah, Commerce
192311006, Gokul, Commerce


event_drama.txt
PARTICIPANTS - Drama
192311002, Bala, Arts


event_music.txt
PARTICIPANTS - Music
192311001, Asha, Arts
192311002, Bala, Arts
192311006, Gokul, Commerce


event_quiz.txt
PARTICIPANTS - Quiz
192311003, Charan, Science
192311005, Farah, Commerce
1

## Automatic Verification Checks

The final cell checks that:
- Duplicate registration is detected.
- Invalid e-mail is rejected.
- Damaged line is rejected.
- Missing department file is recorded.
- Only unique registrations are accepted.
- All output files and event lists are created.


In [3]:
register_numbers = [record[0] for record in master_records]

assert len(register_numbers) == len(set(register_numbers))
assert len(master_records) == 7
assert len(event_participants) > 0

assert (OUTPUT_FOLDER / "master_list.csv").exists()
assert (OUTPUT_FOLDER / "summary_report.txt").exists()
assert (OUTPUT_FOLDER / "error_log.txt").exists()

error_text = (OUTPUT_FOLDER / "error_log.txt").read_text(encoding="utf-8")

assert "Duplicate register number" in error_text
assert "Invalid e-mail address" in error_text
assert "Expected exactly 5 fields" in error_text
assert "management.txt | File not found" in error_text

event_files = list(OUTPUT_FOLDER.glob("event_*.txt"))
assert len(event_files) == len(event_participants)

print("All automatic verification checks passed.")
print("Unique accepted registrations:", len(master_records))
print("Event participant files created:", len(event_files))
print("Duplicate registration: detected")
print("Invalid e-mail: detected")
print("Damaged record: detected")
print("Missing department file: detected and logged")


All automatic verification checks passed.
Unique accepted registrations: 7
Event participant files created: 7
Duplicate registration: detected
Invalid e-mail: detected
Damaged record: detected
Missing department file: detected and logged


## Alternative Concept and Complexity Analysis

### Alternative: Search a list for every earlier registration

Instead of using a set, the program could store accepted register numbers in a list and search that list for every new registration.

```python
seen_numbers = []

if register_number in seen_numbers:
    print("Duplicate registration")
else:
    seen_numbers.append(register_number)
```

### Big-O comparison

Let `n` be the number of registration records.

| Method | Overall time | Reason |
|---|---|---|
| Search a list | O(n²) | Each registration may search many earlier entries. |
| Use a set | O(n) average | Set membership and insertion are O(1) on average. |

**Space complexity:** Both methods use O(n) additional space to store accepted register numbers.

### Conclusion
The set-based design is preferred because it provides faster average duplicate checking. The package separates reading, validation, processing and report writing. It accepts valid unique registrations and records invalid data and missing files without stopping the complete process.
